# Step 3: Semantic Chunking with LLM

This notebook uses a large language model to analyze transcripts and divide them into logical learning sections, each with a title, summary, and precise time boundaries.

**What it does:**
- Reads transcript JSON files from Step 2
- Sends the full transcript to an LLM acting as an Instructional Designer
- The LLM identifies topic boundaries and creates a chunking plan
- Each chunk gets: title, summary, start/end sentence IDs, and start/end timestamps

**Input:** Transcript JSON files from Step 2  
**Output:** `*_plan.json` files, one per video, defining the learning structure

---
## Prerequisites
- Transcript JSON files from `02_transcribe_asr.ipynb`
- An LLM API key (this uses [Kimi K2](https://platform.moonshot.cn/) via OpenAI-compatible endpoint, but any provider works)

> **Tip:** You can swap in any OpenAI-compatible API (GPT-4, Claude, Gemini, Groq, Together, etc.) by changing `BASE_URL`, `API_KEY`, and `MODEL_NAME`.


## Configuration

In [ ]:
import os
import json
import glob
from openai import OpenAI
from google.colab import drive

drive.mount('/content/drive')

# ===============================================
# CONFIGURATION - Edit these values
# ===============================================
INPUT_FOLDER = "/content/drive/MyDrive/your_transcripts/Day_1"
OUTPUT_PLAN_FOLDER = "/content/drive/MyDrive/your_structure/Day_1"

# LLM Provider (OpenAI-compatible endpoint)
MODEL_NAME = "kimi-k2-thinking"                  # Model to use
API_KEY = ""                                      # Your API key
BASE_URL = "https://api.moonshot.ai/v1"           # API base URL
# ===============================================

assert API_KEY, "Please set your LLM API key above."

os.makedirs(OUTPUT_PLAN_FOLDER, exist_ok=True)
client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
print("Configuration loaded.")


## System Prompt

This prompt instructs the LLM to act as an Instructional Designer.  
It produces a JSON array of chunks, each roughly 10 minutes of speaking time, with precise sentence-level boundaries.

Feel free to customize this for your use case (e.g., change max chunk length, output language, or chunking strategy).


In [ ]:
SYSTEM_PROMPT = """You are an expert Instructional Designer.
Your goal is to divide onboarding training transcripts into logical, bite-sized learning chunks (microlearning).
Each chunk should cover a specific topic and generally not exceed 10 minutes of speaking time (using the provided timestamps).
You will receive the full transcript data as a JSON object. You should extract the 'sentences' from `data['transcripts'][0]['sentences']` within the provided JSON and use them to create the learning plan. Each sentence object has 'sentence_id', 'begin_time', 'end_time', and 'text'.

You MUST create chunk titles and summaries in English.
Avoid using any symbols like & or ! in the title because they will become folder paths.

You MUST respond ONLY with valid JSON in the following format. Do not include markdown formatting or explanations.
[
  {
    "title": "Introduction to the AE Role",
    "summary": "Overview of empathy and responsibilities...",
    "start_sentence_id": 0,
    "end_sentence_id": 15,
    "begin_time_ms": 0,
    "end_time_ms": 529812
  },
  ...
]"""

print("System prompt ready.")


## Process Transcripts

Loops through all transcript JSON files, sends each to the LLM, and saves the chunking plan.


In [ ]:
def process_transcript(json_path):
    """Send a transcript to the LLM and save the chunking plan."""
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    basename = os.path.basename(json_path)
    prompt_text = "Here is the full JSON transcript data:\n\n" + json.dumps(data, indent=2)

    print(f"[calling LLM] {basename}...")
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt_text}
        ],
        temperature=0.2
    )

    raw = response.choices[0].message.content.strip()

    # Strip markdown code fences if present
    if raw.startswith("```json"):
        raw = raw[7:]
    if raw.startswith("```"):
        raw = raw[3:]
    if raw.endswith("```"):
        raw = raw[:-3]
    raw = raw.strip()

    try:
        plan = json.loads(raw)
        plan_name = os.path.splitext(basename)[0] + "_plan.json"
        plan_path = os.path.join(OUTPUT_PLAN_FOLDER, plan_name)

        with open(plan_path, 'w', encoding='utf-8') as f:
            json.dump(plan, f, indent=4, ensure_ascii=False)
        print(f"[saved] {plan_name} ({len(plan)} chunks)")
    except json.JSONDecodeError:
        print(f"[error] Failed to parse LLM response for {basename}.")
        print(f"  Response preview: {raw[:500]}")


# Process all transcript files
json_files = sorted(glob.glob(os.path.join(INPUT_FOLDER, "*.json")))
print(f"Found {len(json_files)} transcript(s).\n")

for file_path in json_files:
    process_transcript(file_path)

print("\nSemantic chunking complete.")
